## Homework #7
###  Question 1

My solution (v25.3.9) was determined using the code provided in the question.

###  Question 2

In [ ]:
import pandas as pd
import json
from time import time
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda x: json.dumps(x).encode('utf-8')
)

topic_name = 'green-trips'

# download and process parquet data
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

columns_to_keep = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount'
]

df = pd.read_parquet(url, columns=columns_to_keep)
df = df.sort_values('lpep_pickup_datetime')
df.fillna(0, inplace=True)
df['lpep_pickup_datetime'] = df['lpep_pickup_datetime'].astype(str)
df['lpep_dropoff_datetime'] = df['lpep_dropoff_datetime'].astype(str)
records = df.to_dict(orient='records')

t0 = time()

# send every row
for row in records:
    producer.send(topic_name, value=row)

# force producer to finish sending everything in its buffer
producer.flush()

t1 = time()

print(f'took {(t1 - t0):.2f} seconds')

took 13.73 seconds


As shown by the output above, it took 13.73 seconds to send the data, which is closest to the given answer choice of 10 seconds.

###  Question 3

In [4]:
import json
from kafka import KafkaConsumer

# set up consumer
consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    consumer_timeout_ms=2000
)

long_trips_count = 0
total_messages = 0

# loop through the stream and count
for message in consumer:
    total_messages += 1
    trip_data = message.value
    
    if trip_data.get('trip_distance', 0) > 5.0:
        long_trips_count += 1

print(f"Total messages read: {total_messages}")
print(f"Trips with distance > 5.0: {long_trips_count}")

Total messages read: 49416
Trips with distance > 5.0: 8506


As shown by the output above, there are 8506 trips that have a trip_distance > 5.

###  Question 4

In [ ]:
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import EnvironmentSettings, StreamTableEnvironment

def run_green_aggregation():
    env = StreamExecutionEnvironment.get_execution_environment()
    env.set_parallelism(1)

    settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
    t_env = StreamTableEnvironment.create(env, environment_settings=settings)

    source_ddl = """
        CREATE TABLE green_trips (
            lpep_pickup_datetime VARCHAR,
            lpep_dropoff_datetime VARCHAR,
            PULocationID INT,
            DOLocationID INT,
            passenger_count DOUBLE,
            trip_distance DOUBLE,
            tip_amount DOUBLE,
            total_amount DOUBLE,
            event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
            WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
        ) WITH (
            'connector' = 'kafka',
            'properties.bootstrap.servers' = 'redpanda:29092',
            'topic' = 'green-trips',
            'scan.startup.mode' = 'earliest-offset',
            'properties.auto.offset.reset' = 'earliest',
            'format' = 'json'
        );
    """
    t_env.execute_sql(source_ddl)

    sink_ddl = """
        CREATE TABLE green_trips_aggregated (
            window_start TIMESTAMP(3),
            PULocationID INT,
            num_trips BIGINT,
            PRIMARY KEY (window_start, PULocationID) NOT ENFORCED
        ) WITH (
            'connector' = 'jdbc',
            'url' = 'jdbc:postgresql://postgres:5432/postgres',
            'table-name' = 'green_trips_aggregated',
            'username' = 'postgres',
            'password' = 'postgres',
            'driver' = 'org.postgresql.Driver'
        );
    """
    t_env.execute_sql(sink_ddl)

    try:
        t_env.execute_sql("""
            INSERT INTO green_trips_aggregated
            SELECT
                window_start,
                PULocationID,
                COUNT(*) AS num_trips
            FROM TABLE(
                TUMBLE(TABLE green_trips, DESCRIPTOR(event_timestamp), INTERVAL '5' MINUTE)
            )
            GROUP BY window_start, PULocationID;
        """).wait()
    except Exception as e:
        print("Writing records to JDBC failed:", str(e))

if __name__ == '__main__':
    run_green_aggregation()

My solution (74) was determined using the Flink job code above as well as the SQL code provided in the question.

###  Question 5

In [ ]:
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import EnvironmentSettings, StreamTableEnvironment

def run_green_sessions():
    env = StreamExecutionEnvironment.get_execution_environment()
    env.set_parallelism(1)

    settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
    t_env = StreamTableEnvironment.create(env, environment_settings=settings)

    source_ddl = """
        CREATE TABLE green_trips (
            lpep_pickup_datetime VARCHAR,
            lpep_dropoff_datetime VARCHAR,
            PULocationID INT,
            DOLocationID INT,
            passenger_count DOUBLE,
            trip_distance DOUBLE,
            tip_amount DOUBLE,
            total_amount DOUBLE,
            event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
            WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
        ) WITH (
            'connector' = 'kafka',
            'properties.bootstrap.servers' = 'redpanda:29092',
            'topic' = 'green-trips',
            'scan.startup.mode' = 'earliest-offset',
            'properties.auto.offset.reset' = 'earliest',
            'format' = 'json'
        );
    """
    t_env.execute_sql(source_ddl)

    sink_ddl = """
        CREATE TABLE green_trips_sessions (
            window_start TIMESTAMP(3),
            window_end TIMESTAMP(3),
            PULocationID INT,
            num_trips BIGINT,
            PRIMARY KEY (window_start, window_end, PULocationID) NOT ENFORCED
        ) WITH (
            'connector' = 'jdbc',
            'url' = 'jdbc:postgresql://postgres:5432/postgres',
            'table-name' = 'green_trips_sessions',
            'username' = 'postgres',
            'password' = 'postgres',
            'driver' = 'org.postgresql.Driver'
        );
    """
    t_env.execute_sql(sink_ddl)

    try:
        t_env.execute_sql("""
            INSERT INTO green_trips_sessions
            SELECT
                window_start,
                window_end,
                PULocationID,
                COUNT(*) AS num_trips
            FROM TABLE(
                SESSION(TABLE green_trips PARTITION BY PULocationID, DESCRIPTOR(event_timestamp), INTERVAL '5' MINUTE)
            )
            GROUP BY window_start, window_end, PULocationID;
        """).wait()
    except Exception as e:
        print("Writing records to JDBC failed:", str(e))

if __name__ == '__main__':
    run_green_sessions()

My solution was determined using the Flink job code above as well as SQL code I generated; please see README. I received a solution of 82, which is closest to the given answer choice of 81.

###  Question 6

In [ ]:
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import EnvironmentSettings, StreamTableEnvironment

def run_hourly_tips():

    env = StreamExecutionEnvironment.get_execution_environment()
    env.set_parallelism(1)
    env.enable_checkpointing(10 * 1000)

    settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
    t_env = StreamTableEnvironment.create(env, environment_settings=settings)

    source_ddl = """
        CREATE TABLE green_trips (
            lpep_pickup_datetime VARCHAR,
            lpep_dropoff_datetime VARCHAR,
            PULocationID INT,
            DOLocationID INT,
            passenger_count DOUBLE,
            trip_distance DOUBLE,
            tip_amount DOUBLE,
            total_amount DOUBLE,
            event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
            WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
        ) WITH (
            'connector' = 'kafka',
            'properties.bootstrap.servers' = 'redpanda:29092',
            'topic' = 'green-trips',
            'scan.startup.mode' = 'earliest-offset',
            'properties.auto.offset.reset' = 'earliest',
            'format' = 'json'
        );
    """
    t_env.execute_sql(source_ddl)

    sink_ddl = """
        CREATE TABLE green_trips_tip_hourly (
            window_start TIMESTAMP(3),
            total_tips DOUBLE,
            PRIMARY KEY (window_start) NOT ENFORCED
        ) WITH (
            'connector' = 'jdbc',
            'url' = 'jdbc:postgresql://postgres:5432/postgres',
            'table-name' = 'green_trips_tip_hourly',
            'username' = 'postgres',
            'password' = 'postgres',
            'driver' = 'org.postgresql.Driver'
        );
    """
    t_env.execute_sql(sink_ddl)

    try:
        print("Submitting the 1-hour tumbling window job for total tips...")
        t_env.execute_sql("""
            INSERT INTO green_trips_tip_hourly
            SELECT
                window_start,
                SUM(tip_amount) AS total_tips
            FROM TABLE(
                TUMBLE(TABLE green_trips, DESCRIPTOR(event_timestamp), INTERVAL '1' HOUR)
            )
            GROUP BY window_start;
        """).wait()
    except Exception as e:
        print("Writing records to JDBC failed:", str(e))

if __name__ == '__main__':
    run_hourly_tips()

My solution (2025-10-16 18:00:00) was determined using the Flink job code above as well as SQL code I generated; please see README.